# Dense retrieval (SapBERT, BioLORD-2023)

Computes dense (embedding) retrieval results for TermMap on a GPU runtime, one set per encoder.
Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

All files are read from and written to one Google Drive folder, `My Drive/HiLabs_Clinical`.

**Inputs** (place in the folder root before running):

| File | Where it comes from |
|---|---|
| `terminology_ICD10CM.desc.parquet` | `artifacts/` |
| `terminology_SNOMEDCT_US.desc.parquet` | `artifacts/` |
| `dense_queries.txt` | `outputs/`, written by `python -m termmap export-queries` |

**Outputs** in `outputs/`, one pair per encoder. Copy them into the repository's `artifacts/`.

| Encoder | Files |
|---|---|
| SapBERT | `dense_hits_ICD10CM.parquet`, `dense_hits_SNOMEDCT_US.parquet` |
| BioLORD-2023 | `dense_hits_biolord_ICD10CM.parquet`, `dense_hits_biolord_SNOMEDCT_US.parquet` |

Description embeddings are cached in `cache/`, so a later run with a new query file only encodes the queries.
An encoder whose outputs already cover every query in `dense_queries.txt` is skipped.

In [ ]:
!pip -q install sentence-transformers pyarrow

In [ ]:
import os, numpy as np, pandas as pd, torch
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/HiLabs_Clinical"
assert os.path.isdir(WORK), f"create the folder {WORK} in Google Drive first"
CACHE, OUT = f"{WORK}/cache", f"{WORK}/outputs"
os.makedirs(CACHE, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), "switch the runtime to a GPU"
torch.manual_seed(0)
# name -> (model, output file prefix)
ENCODERS = {
    "sapbert": ("cambridgeltl/SapBERT-from-PubMedBERT-fulltext", "dense_hits"),
    "biolord": ("FremyCompany/BioLORD-2023", "dense_hits_biolord"),
}
SYSTEMS = ["ICD10CM", "SNOMEDCT_US"]
K = 40          # distinct codes kept per query

In [ ]:
needed = ["terminology_ICD10CM.desc.parquet", "terminology_SNOMEDCT_US.desc.parquet", "dense_queries.txt"]
missing = [f for f in needed if not os.path.exists(f"{WORK}/{f}")]
assert not missing, f"missing in {WORK}: {missing}"
queries = [q for q in open(f"{WORK}/dense_queries.txt", encoding="utf-8").read().split("\n") if q]
print(len(queries), "queries")

In [ ]:
def encode(model, texts):
    return model.encode(texts, batch_size=512, normalize_embeddings=True,
                        show_progress_bar=True, convert_to_tensor=True)

def description_embeddings(name, model, system):
    """Load cached embeddings when they match the current description table."""
    d = pd.read_parquet(f"{WORK}/terminology_{system}.desc.parquet")
    codes = d["code"].to_numpy().astype(str)
    tag = "dense" if name == "sapbert" else f"dense_{name}"
    emb_path, code_path = f"{CACHE}/{tag}_{system}.emb.npy", f"{CACHE}/{tag}_{system}.codes.npy"
    if os.path.exists(emb_path) and os.path.exists(code_path):
        cached_codes = np.load(code_path)
        if len(cached_codes) == len(codes) and (cached_codes == codes).all():
            print(name, system, "using cached embeddings")
            return torch.from_numpy(np.load(emb_path)).cuda(), codes
    print(name, system, "encoding", len(d), "descriptions")
    e = encode(model, d["norm"].tolist()).half()
    np.save(emb_path, e.cpu().numpy())
    np.save(code_path, codes)
    return e, codes

def covered(prefix):
    want = set(queries)
    for system in SYSTEMS:
        p = f"{OUT}/{prefix}_{system}.parquet"
        if not os.path.exists(p) or not want <= set(pd.read_parquet(p, columns=["query"])["query"]):
            return False
    return True

In [ ]:
for name, (model_name, prefix) in ENCODERS.items():
    if covered(prefix):
        print(name, "outputs already cover all queries, skipping")
        continue
    model = SentenceTransformer(model_name, device="cuda")
    q = encode(model, queries).half()
    for system in SYSTEMS:
        E, C = description_embeddings(name, model, system)
        rows = []
        for start in range(0, len(queries), 1024):
            sims = q[start:start + 1024] @ E.T
            val, idx = torch.topk(sims.float(), k=K * 4, dim=1)
            val, idx = val.cpu().numpy(), idx.cpu().numpy()
            for i in range(len(idx)):
                seen, r = set(), 0
                for s, j in zip(val[i], idx[i]):
                    c = C[j]
                    if c in seen:
                        continue
                    seen.add(c)
                    rows.append((queries[start + i], r, c, float(s)))
                    r += 1
                    if r == K:
                        break
        out = pd.DataFrame(rows, columns=["query", "rank", "code", "score"])
        path = f"{OUT}/{prefix}_{system}.parquet"
        out.to_parquet(path, index=False)
        print(name, system, out.shape, out["query"].nunique(), "queries ->", path)
        del E
        torch.cuda.empty_cache()
    del model, q
    torch.cuda.empty_cache()